In [1]:
# Load necessary library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/ADEGuard/final_dataset_2020_2025.csv', low_memory=False)

In [4]:
df.shape

(1228214, 31)

In [5]:
df.columns

Index(['VAERS_ID', 'STATE', 'SEX', 'AGE_YRS', 'RECVDATE', 'RPT_DATE',
       'VAX_DATE', 'ONSET_DATE', 'DIED', 'L_THREAT', 'ER_VISIT', 'HOSPITAL',
       'DISABLE', 'RECOVD', 'OFC_VISIT', 'ER_ED_VISIT', 'BIRTH_DEFECT',
       'HOSPDAYS', 'NUMDAYS', 'SYMPTOM_TEXT', 'LAB_DATA', 'OTHER_MEDS',
       'CUR_ILL', 'HISTORY', 'PRIOR_VAX', 'ALLERGIES', 'ALL_SYMPTOMS',
       'VAX_NAME', 'VAX_TYPE', 'VAX_MANU', 'YEAR'],
      dtype='object')

In [6]:
df.head()

,VAERS_ID,STATE,SEX,AGE_YRS,RECVDATE,RPT_DATE,VAX_DATE,ONSET_DATE,DIED,L_THREAT,...,OTHER_MEDS,CUR_ILL,HISTORY,PRIOR_VAX,ALLERGIES,ALL_SYMPTOMS,VAX_NAME,VAX_TYPE,VAX_MANU,YEAR
0,275438,CA,F,NaN,01/13/2020,NaN,03/12/2007,03/26/2007,1,0,...,YASMIN,NaN,Medical History/Concurrent Conditions: Non-smo...,NaN,NaN,"['Pulmonary congestion', 'Death', 'Autopsy', '...",['HPV (GARDASIL)'],['HPV4'],['MERCK & CO. INC.'],2020
1,340381,PA,M,11.0,07/29/2020,NaN,02/13/2009,02/13/2009,0,0,...,INVEGA,Allergic rhinitis; Asthma (Mild persistent); A...,NaN,NaN,NaN,"['Injection site erythema', 'Type III immune c...","['MENINGOCOCCAL CONJUGATE (MENACTRA)', 'TDAP (...","['MNQ', 'TDAP']","['SANOFI PASTEUR', 'UNKNOWN MANUFACTURER']",2020
2,361824,NY,F,NaN,11/19/2020,NaN,NaN,NaN,0,0,...,NaN,Drug allergy; Penicillin allergy,NaN,NaN,NaN,"['Urticaria', 'Vaccination complication', 'Rash']",['INFLUENZA (SEASONAL) (FLUZONE HIGH-DOSE QUAD...,['FLU4'],['SANOFI PASTEUR'],2020
3,371224,CA,U,NaN,10/19/2020,NaN,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,['Facial paralysis'],['ZOSTER LIVE (ZOSTAVAX)'],['VARZOS'],['MERCK & CO. INC.'],2020
4,386005,NM,F,NaN,01/13/2020,NaN,04/12/2010,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,['Post herpetic neuralgia'],['ZOSTER LIVE (ZOSTAVAX)'],['VARZOS'],['MERCK & CO. INC.'],2020


In [7]:
type(df['VAX_TYPE'].iloc[0])

str

In [8]:
# filter Covid data to create new dataset
df = df[df["VAX_TYPE"].str.contains("COVID", na=False)]

In [9]:
df.shape

(1029334, 31)

In [10]:
#check the missing value
print(df["SYMPTOM_TEXT"].isnull().sum())

1479


In [11]:
#drop the null value
df = df.dropna(subset=["SYMPTOM_TEXT"])

In [12]:
df.shape

(1027855, 31)

In [13]:
# text lenght from SYMPTOM_TEXT
df["TEXT_LEN"] = df["SYMPTOM_TEXT"].str.len()

In [14]:
df["TEXT_LEN"].describe()

,TEXT_LEN
count,1.027855e+06
mean,7.162373e+02
std,1.028613e+03
min,1.000000e+00
25%,1.000000e+02
50%,3.060000e+02
75%,1.031000e+03
max,6.399600e+04


In [15]:
#approximate token count
df["APPROX_TOKENS"] = df["TEXT_LEN"] / 4

In [16]:
df[df["APPROX_TOKENS"] > 2000].shape

(1995, 33)

In [17]:
# filter 1995 rows and create new dataset
df = df[df["APPROX_TOKENS"] <= 2000]

In [18]:
df.shape

(1025860, 33)

In [19]:
#check null value
print(df["AGE_YRS"].isnull().sum())

104982


In [20]:
type(df['AGE_YRS'].iloc[0])

numpy.float64

In [21]:
df["AGE_YRS"].describe()

,AGE_YRS
count,920878.000000
mean,50.119285
std,20.334999
min,0.080000
25%,35.000000
50%,51.000000
75%,66.000000
max,119.000000


In [22]:
#create age gorup
def age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age < 18:
        return "Child"
    elif age < 35:
        return "Young_Adult"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

df["AGE_GROUP"] = df["AGE_YRS"].apply(age_group)

In [23]:
df["AGE_GROUP"].value_counts()

,count
AGE_GROUP,
Adult,359956
Senior,336504
Young_Adult,160484
Unknown,104982
Child,63934


In [24]:
df["SEX"].value_counts(dropna=False)

,count
SEX,
F,652888
M,329302
U,43433
NaN,237


In [25]:
df["SEX"] = df["SEX"].fillna("Unknown")

In [26]:
df["SEX"] = df["SEX"].replace("U", "Unknown")

In [27]:
df["SEX"].value_counts()

,count
SEX,
F,652888
M,329302
Unknown,43670


In [28]:
cols = ["DIED", "HOSPITAL", "L_THREAT", "DISABLE"]

for c in cols:
    print(c)
    print(df[c].value_counts(dropna=False))
    print()

DIED
DIED
0    1006087
1      19773
Name: count, dtype: int64

HOSPITAL
HOSPITAL
0    933864
1     91996
Name: count, dtype: int64

L_THREAT
L_THREAT
0    1009473
1      16387
Name: count, dtype: int64

DISABLE
DISABLE
0    1005886
1      19974
Name: count, dtype: int64



In [29]:
#create flag count
df["FLAG_COUNT"] = (
    df["DIED"] +
    df["HOSPITAL"] +
    df["L_THREAT"] +
    df["DISABLE"]
)

In [30]:
df["FLAG_COUNT"].value_counts()

,count
FLAG_COUNT,
0,905111
1,97177
2,19904
3,3527
4,141


In [31]:
# create severity level
def map_severity(row):

    if row["DIED"] == 1:
        return "Severe"

    elif row["L_THREAT"] == 1:
        return "Severe"

    elif row["DISABLE"] == 1:
        return "Severe"

    elif row["HOSPITAL"] == 1:
        return "Moderate"

    else:
        return "Mild"

df["SEVERITY_LEVEL"] = df.apply(map_severity, axis=1)

In [32]:
df["SEVERITY_LEVEL"].value_counts()

,count
SEVERITY_LEVEL,
Mild,905111
Moderate,69481
Severe,51268


In [33]:
df.groupby("SEVERITY_LEVEL")[
    ["DIED","L_THREAT","DISABLE","HOSPITAL"]
].mean()

,DIED,L_THREAT,DISABLE,HOSPITAL
SEVERITY_LEVEL,,,,
Mild,0.000000,0.000000,0.0000,0.000000
Moderate,0.000000,0.000000,0.0000,1.000000
Severe,0.385679,0.319634,0.3896,0.439163


In [34]:
#create a dataset for model
df_model = df[
    ["SYMPTOM_TEXT", "SEX", "AGE_GROUP", "SEVERITY_LEVEL"]
]

In [35]:
df_model.shape

(1025860, 4)

In [36]:
df_model.to_csv("/content/drive/MyDrive/ADEGuard/vaers_clean_dataset.csv", index=False)